# [Chapter 9 — Practical Issues in Fitting, §9.3] Pitfall 3: Diagnosing and Resolving Practical Identifiability Failures

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 9 — Practical Issues in Fitting
**Considerations developed:** 8, 9
**Estimated runtime:** ≤ 1m

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/)

## What this notebook does
Profile-likelihood approach to diagnosing practical non-identifiability. Shows how a flat profile in one direction signals that the data cannot constrain that parameter.

## What you should already know
Chapter 8 fitting + Ch 8.5 F_0 fitting.


## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
import numpy as np
import matplotlib.pyplot as plt
from shared import book_style, BOOK_COLORS, integrate_sir_i
from shared.parameters import baseline_chapter_08
from shared.seeds import set_seed_chapter_08
from shared.verification import assert_within_tolerance
set_seed_chapter_08()
book_style()


## Profile likelihood

For each candidate value of a target parameter (e.g., $\mathcal{R}_0$), fit all other parameters to the data while holding the target fixed. Plot $-2 \log L$ as a function of the target. A narrow well = identifiable; a flat profile = non-identifiable.

In [ ]:
from scipy.optimize import minimize

params = baseline_chapter_08()
true_R0 = params['c_I'] * params['beta'] * params['tau_R']

# Generate data
result = integrate_sir_i(params)
t, S, I = result['t'], result['S'], result['I']
J_obs = np.maximum(params['c_I'] * params['beta'] * (S/params['N']) * I + 0.05 * (params['c_I'] * params['beta'] * (S/params['N']) * I).max() * np.random.randn(len(t)), 0)

def fit_for_R0(R0_fixed, F0_init=0.95):
    # Profile likelihood: fix R_0, optimize over F_0
    def loss(F0):
        F0 = F0[0]
        if F0 <= 0.1 or F0 >= 1.0: return 1e10
        p = params.copy()
        p['c_I'] = R0_fixed / (params['beta'] * params['tau_R'])
        p['S0'] = F0
        p['I0'] = 0.001
        p['R0_init'] = 1.0 - F0 - 0.001
        res = integrate_sir_i(p)
        J_pred = p['c_I'] * p['beta'] * (res['S']/p['N']) * res['I']
        return np.sum((J_pred - J_obs)**2)
    r = minimize(loss, [F0_init], method='Nelder-Mead', options={'xatol':1e-3})
    return r.fun, r.x[0]

R0_test = np.linspace(1.5, 2.5, 11)
profile = [fit_for_R0(r) for r in R0_test]
losses = [p[0] for p in profile]
F0_optima = [p[1] for p in profile]

losses_normalized = np.array(losses) - min(losses)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ax = axes[0]
ax.plot(R0_test, losses_normalized, 'o-', color=BOOK_COLORS['primary'])
ax.axhline(3.84, ls='--', color=BOOK_COLORS['negative'], alpha=0.6, label='95% CI threshold (chi^2_1)')
ax.axvline(true_R0, ls=':', color=BOOK_COLORS['neutral'], alpha=0.5, label='True R_0')
ax.set_xlabel(r'$R_0$ (fixed)')
ax.set_ylabel(r'$-2 \log L$ - min')
ax.set_title('Profile likelihood (well-identifiable: narrow well)')
ax.legend()

ax = axes[1]
ax.plot(R0_test, F0_optima, 'o-', color=BOOK_COLORS['lambda_hat'])
ax.set_xlabel(r'$R_0$ (fixed)')
ax.set_ylabel(r'Optimal $F_0$ at this $R_0$')
ax.set_title('Compensating parameter (correlation diagnostic)')
plt.tight_layout()
plt.show()
